Fake News Classifier using lstm

In [2]:
import pandas as pd

In [3]:
data = pd.read_csv("data.csv")

In [4]:
data.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [5]:
data.drop(columns="date", axis=1, inplace=True)

In [6]:
data.head()

,title,text,subject,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,1


In [7]:
data.tail()

,title,text,subject,label
44893,UNREAL! CBS’S TED KOPPEL Tells Sean Hannity He...,,politics,0
44894,PM May seeks to ease Japan's Brexit fears duri...,LONDON/TOKYO (Reuters) - British Prime Ministe...,worldnews,1
44895,Merkel: Difficult German coalition talks can r...,BERLIN (Reuters) - Chancellor Angela Merkel sa...,worldnews,1
44896,Trump Stole An Idea From North Korean Propaga...,Jesus f*cking Christ our President* is a moron...,News,0
44897,BREAKING: HILLARY CLINTON’S STATE DEPARTMENT G...,IF SHE S NOT TOAST NOW THEN WE RE IN BIGGER TR...,politics,0


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   label    44898 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.4+ MB


In [9]:
data.isnull().sum()

title      0
text       0
subject    0
label      0
dtype: int64

In [10]:
data = data.dropna()

In [11]:
data.shape

(44898, 4)

In [12]:
X = data.drop('label', axis=1)

In [13]:
Y = data['label']

In [14]:
import tensorflow as tf

In [15]:
tf.__version__

'2.21.0'

In [16]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM

In [17]:
### Vocab size

voc_size = 10000

One Hot Representation

In [18]:
messages = X.copy()

In [19]:
messages['title'][1]

'Trump drops Steve Bannon from National Security Council'

In [20]:
messages.reset_index(inplace=True)

In [21]:
import nltk
import re
from nltk.corpus import stopwords

In [22]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [23]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()
    
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [24]:
corpus

['ben stein call th circuit court commit coup tat constitut',
 'trump drop steve bannon nation secur council',
 'puerto rico expect u lift jone act ship restrict',
 'oop trump accident confirm leak isra intellig russia video',
 'donald trump head scotland reopen golf resort',
 'paul ryan respond dem sit gun control disgust way video',
 'awesom diamond silk rip press believ video',
 'stand cheer ukip parti leader slam germani franc eu invas phoni refuge video',
 'north korea show sign seriou talk u offici',
 'trump signal willing rais u minimum wage',
 'new jersey christi mull run lead republican parti report',
 'hillari clinton spot dine alon',
 'franc germani want iran revers ballist missil program',
 'aid eu commiss head tweet pictur white smoke brexit meet may',
 'trump issu warn man armi could isi video',
 'u give lao extra million help clear unexplod ordnanc',
 'judg declar babi name illeg prevent emot harm',
 'paul ryan take monument humili photo constitu expertli troll imag',
 '

In [25]:
one_hot_repr = [one_hot(words, voc_size) for words in corpus]
one_hot_repr

[[5367, 3992, 726, 2268, 3822, 8336, 1898, 8833, 8474, 1228],
 [176, 577, 6512, 5378, 903, 7844, 8789],
 [1301, 877, 9601, 3074, 2976, 394, 223, 9573, 186],
 [3727, 176, 9671, 4108, 4911, 8946, 3510, 2439, 6695],
 [3365, 176, 3813, 5893, 5198, 4074, 6482],
 [3076, 9360, 4990, 6716, 9535, 4033, 9068, 8969, 7763, 6695],
 [2136, 3579, 6043, 117, 7241, 3099, 6695],
 [6505, 5396, 6194, 1575, 9739, 1472, 7971, 4940, 9573, 1564, 6825, 480, 6695],
 [4553, 3722, 9287, 646, 1619, 5992, 3074, 8452],
 [176, 9807, 5857, 3431, 3074, 3242, 7728],
 [9817, 4086, 878, 7853, 7910, 808, 9211, 1575, 3877],
 [9445, 5115, 1273, 6888, 9158],
 [4940, 7971, 9720, 3248, 8701, 1863, 2785, 9825],
 [5897, 9573, 197, 3813, 6312, 4995, 9005, 1869, 8633, 5594, 7591],
 [176, 9761, 2764, 4401, 7249, 3582, 8459, 6695],
 [3074, 9854, 909, 8525, 2427, 233, 546, 6629, 953],
 [3564, 2665, 6844, 5903, 8127, 1970, 6985, 7575],
 [3076, 9360, 6283, 2276, 5274, 7402, 691, 1275, 3805, 8791],
 [9211, 6888, 176, 5072, 7836, 7784, 54

Embedding Representation

In [26]:
sent_length = 20
embedded_docs = pad_sequences(one_hot_repr, padding='pre' , maxlen=sent_length)
embedded_docs

array([[   0,    0,    0, ..., 8833, 8474, 1228],
       [   0,    0,    0, ...,  903, 7844, 8789],
       [   0,    0,    0, ...,  223, 9573,  186],
       ...,
       [   0,    0,    0, ..., 5992, 1518, 2951],
       [   0,    0,    0, ..., 5120, 5111, 9818],
       [   0,    0,    0, ..., 3637, 2530, 1703]],
      shape=(44898, 20), dtype=int32)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = Sequential([
    Input(shape=(sent_length,)),
    Embedding(voc_size, 40),
    LSTM(100,
        dropout=0.3,
        recurrent_dropout=0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 40)         │       400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 456,501 (1.74 MB)

 Trainable params: 456,501 (1.74 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
len(embedded_docs), Y.shape

(44898, (44898,))

In [29]:
import numpy as np
X_final = np.array(embedded_docs)
Y_final = np.array(Y)

In [30]:
X_final.shape, Y_final.shape

((44898, 20), (44898,))

In [31]:
from sklearn.model_selection import train_test_split

X_train , X_test , Y_train , Y_test = train_test_split(X_final, Y_final, test_size=0.2, random_state=42)

In [32]:
## Model traning 
model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=10, batch_size=62)

Epoch 1/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 20s 26ms/step - accuracy: 0.9070 - loss: 0.2296 - val_accuracy: 0.9504 - val_loss: 0.1330
Epoch 2/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9547 - loss: 0.1184 - val_accuracy: 0.9560 - val_loss: 0.1207
Epoch 3/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9658 - loss: 0.0896 - val_accuracy: 0.9541 - val_loss: 0.1209
Epoch 4/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 15s 25ms/step - accuracy: 0.9739 - loss: 0.0697 - val_accuracy: 0.9543 - val_loss: 0.1304
Epoch 5/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9789 - loss: 0.0560 - val_accuracy: 0.9543 - val_loss: 0.1436
Epoch 6/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.9819 - loss: 0.0487 - val_accuracy: 0.9510 - val_loss: 0.1504
Epoch 7/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9867 - loss: 0.0368 - val_accuracy: 0.9503 - val_loss: 0.1727
Epoch 8/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.9885 - loss: 0.0321 - 

In [33]:
y_pred = model.predict(X_test)


281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step


In [36]:
y_pred = np.where(y_pred > 0.5,1,0)

In [37]:
from sklearn.metrics import confusion_matrix


In [38]:
confusion_matrix(Y_test, y_pred)



array([[4506,  204],
       [ 262, 4008]])

In [39]:
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, y_pred)

0.9481069042316258

In [40]:
from sklearn.metrics import classification_report
print(classification_report(Y_test , y_pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95      4710
           1       0.95      0.94      0.95      4270

    accuracy                           0.95      8980
   macro avg       0.95      0.95      0.95      8980
weighted avg       0.95      0.95      0.95      8980

